# 📊 EMV Chunk Dataset — Statistics & Quality Analysis
### Book 1 · Adaptive Chunks with Metadata

This notebook loads the pre-computed EMV chunk JSON file and produces a comprehensive set of statistics, grouped analyses, quality flags, and visualisations — all saved to a local output folder.

## 1. Imports & Configuration

In [1]:
import json
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for scripts
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH  = Path("../../../data/processed/book1/done/adaptive_chunks.json")
OUTPUT_DIR  = Path("../../../data/processed/book1/done/analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi"      : 130,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "axes.grid"       : True,
    "grid.alpha"      : 0.3,
    "font.size"       : 11,
})

print(f"✅ Output directory: {OUTPUT_DIR.resolve()}")

✅ Output directory: /app/src/data/processed/book1/done/analysis


## 2. Load Data

In [2]:
def load_chunks(path: Path) -> list:
    """Load chunks from a JSON file. Supports top-level list or dict wrapper."""
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    for key in ("chunks", "data", "items", "results"):
        if key in data:
            return data[key]
    raise ValueError("Unexpected JSON structure — expected a list of chunk objects.")


raw_chunks = load_chunks(INPUT_PATH)
print(f"✅ Loaded {len(raw_chunks):,} chunks")
print("   Sample keys:", list(raw_chunks[0].keys()) if raw_chunks else "(empty)")

✅ Loaded 78 chunks
   Sample keys: ['chunk_id', 'section_id', 'chunk_index', 'doc_id', 'title', 'section_number', 'page_num', 'level', 'parent_titles', 'parent_section_id', 'child_section_ids', 'text', 'token_count', 'type', 'source_block_ids', 'block_count', 'is_split', 'split_group_id', 'overlap_prev_tokens', 'overlap_next_tokens', 'candidate_source', 'chunking_reason', 'size_band', 'oversize_reason', 'merged_titles', 'merged_from_chunk_ids', 'merged_from_section_ids', 'merge_group_size', 'page_span']


## 3. Build DataFrame

In [3]:
def build_dataframe(chunks: list) -> pd.DataFrame:
    """
    Convert raw chunk list to a tidy DataFrame.
    Derived columns are added here for convenience.
    """
    df = pd.DataFrame(chunks)

    # ── Ensure expected columns exist (fill missing with sensible defaults) ─
    defaults = {
        "chunk_id"           : "",
        "section_id"         : "",
        "chunk_index"        : "1/1",
        "doc_id"             : "",
        "title"              : "",
        "section_number"     : "",
        "page_num"           : None,
        "level"              : 0,
        "parent_titles"      : None,
        "parent_section_id"  : "",
        "child_section_ids"  : None,
        "text"               : "",
        "token_count"        : 0,
        "type"               : "text",
        "source_block_ids"   : None,
        "block_count"        : 0,
        "is_split"           : False,
        "split_group_id"     : None,
        "overlap_prev_tokens": 0,
        "overlap_next_tokens": 0,
        "candidate_source"   : "",
        "chunking_reason"    : "",
        "size_band"          : "",
        "oversize_reason"    : None,
    }
    for col, default in defaults.items():
        if col not in df.columns:
            df[col] = default

    # ── Fill nulls safely ─────────────────────────────────────────────────
    df["token_count"]        = pd.to_numeric(df["token_count"], errors="coerce").fillna(0).astype(int)
    df["block_count"]        = pd.to_numeric(df["block_count"], errors="coerce").fillna(0).astype(int)
    df["overlap_prev_tokens"]= pd.to_numeric(df["overlap_prev_tokens"], errors="coerce").fillna(0).astype(int)
    df["overlap_next_tokens"]= pd.to_numeric(df["overlap_next_tokens"], errors="coerce").fillna(0).astype(int)
    df["is_split"]           = df["is_split"].fillna(False).astype(bool)
    df["text"]               = df["text"].fillna("").astype(str)
    df["level"]              = pd.to_numeric(df["level"], errors="coerce").fillna(0).astype(int)

    # ── Derived columns ───────────────────────────────────────────────────
    df["char_count"]           = df["text"].str.len()
    df["char_token_ratio"]     = df.apply(
        lambda r: round(r["char_count"] / r["token_count"], 2) if r["token_count"] > 0 else 0, axis=1
    )
    df["parent_titles_len"]    = df["parent_titles"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    df["child_count"]          = df["child_section_ids"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    df["is_leaf_source"]       = df["child_count"] == 0
    df["source_block_ids_len"] = df["source_block_ids"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    # Denominator of chunk_index (e.g. '2/5' → 5)
    df["chunk_index_denom"] = df["chunk_index"].apply(
        lambda x: int(str(x).split("/")[-1]) if "/" in str(x) else 1
    )

    return df


df = build_dataframe(raw_chunks)
print(f"✅ DataFrame built — shape: {df.shape}")
df.head(2)

✅ DataFrame built — shape: (78, 38)


,chunk_id,section_id,chunk_index,doc_id,title,section_number,page_num,level,parent_titles,parent_section_id,...,page_span,table_id,table_title,char_count,char_token_ratio,parent_titles_len,child_count,is_leaf_source,source_block_ids_len,chunk_index_denom
0,EMV_Book_1_v4.4_10_PartI.1.1_text_1__merged__E...,EMV_Book_1_v4.4_10_PartI.1.1,merged_2,EMV_Book_1,Changes in Version 4.4,PartI.1.1,10,2,"[General, Scope]",EMV_Book_1_v4.4_10_PartI.1,...,"[10, 10]",NaN,NaN,1790,4.85,2,0,True,16,1
1,EMV_Book_1_v4.4_10_PartI.1_text_1,EMV_Book_1_v4.4_10_PartI.1,1/1,EMV_Book_1,Scope,PartI.1,10,1,[General],EMV_Book_1_v4.4_9_Part_I,...,NaN,NaN,NaN,808,5.53,1,4,False,3,1


## 4. A · Basic Dataset Statistics

In [4]:
def compute_basic_stats(df: pd.DataFrame) -> dict:
    """Compute top-level descriptive statistics for the chunk dataset."""
    tc = df["token_count"]
    stats = {
        "total_chunks"         : int(len(df)),
        "text_chunks"          : int((df["type"] == "text").sum()),
        "table_chunks"         : int((df["type"] == "table").sum()),
        "unique_sections"      : int(df["section_id"].nunique()),
        "unique_documents"     : int(df["doc_id"].nunique()),
        "token_count_mean"     : round(float(tc.mean()), 2),
        "token_count_median"   : round(float(tc.median()), 2),
        "token_count_min"      : int(tc.min()),
        "token_count_max"      : int(tc.max()),
        "token_count_std"      : round(float(tc.std()), 2),
        "total_tokens"         : int(tc.sum()),
    }
    return stats


basic = compute_basic_stats(df)

print("\n" + "═" * 45)
print("  A · BASIC STATS")
print("═" * 45)
for k, v in basic.items():
    print(f"  {k:<28}: {v:>10,}" if isinstance(v, int) else f"  {k:<28}: {v:>10}")


═════════════════════════════════════════════
  A · BASIC STATS
═════════════════════════════════════════════
  total_chunks                :         78
  text_chunks                 :         57
  table_chunks                :         21
  unique_sections             :         42
  unique_documents            :          1
  token_count_mean            :     270.26
  token_count_median          :      228.0
  token_count_min             :         13
  token_count_max             :        586
  token_count_std             :     183.74
  total_tokens                :     21,080


## 5. B · Size-Band Analysis

In [5]:
def compute_size_band_stats(df: pd.DataFrame) -> dict:
    """Count chunks in each token size band and compute percentages."""
    total = len(df)
    bands = {
        "below_100"       : int((df["token_count"] < 100).sum()),
        "between_100_179"       : int(((df["token_count"] >= 100) & (df["token_count"] < 180)).sum()),
        "between_180_349" : int(((df["token_count"] >= 180) & (df["token_count"] < 350)).sum()),
        "between_350_550" : int(((df["token_count"] >= 350) & (df["token_count"] <= 550)).sum()),
        "above_550"       : int((df["token_count"] > 550).sum()),
    }
    # Use the size_band column if it exists, else fall back to recomputed counts
    if df["size_band"].notna().any():
        band_counts = df["size_band"].value_counts().to_dict()
        for k, v in band_counts.items():
            if k in bands:
                bands[k] = int(v)

    result = {}
    for band, count in bands.items():
        result[band] = {"count": count, "pct": round(100 * count / total, 1) if total else 0}
    return result


size_bands = compute_size_band_stats(df)

print("\n B · SIZE BANDS")
print(f"  {'Band':<22} {'Count':>8} {'%':>8}")
print("  " + "-" * 40)
for band, vals in size_bands.items():
    print(f"  {band:<22} {vals['count']:>8,} {vals['pct']:>7.1f}%")


 B · SIZE BANDS
  Band                      Count        %
  ----------------------------------------
  below_100                    18    23.1%
  between_100_179              15    19.2%
  between_180_349              20    25.6%
  between_350_550              20    25.6%
  above_550                     5     6.4%


## 6. C · Split Analysis

In [6]:
def compute_split_stats(df: pd.DataFrame) -> dict:
    """Analyse how many chunks came from splitting large sections."""
    split_df = df[df["is_split"] == True]
    non_split = df[df["is_split"] == False]

    # Unique split groups
    split_groups = split_df["split_group_id"].dropna()
    group_sizes  = split_groups.value_counts()

    # Denominator distribution
    denom_dist = df["chunk_index_denom"].value_counts().sort_index().to_dict()

    return {
        "split_chunks"            : int(len(split_df)),
        "non_split_chunks"        : int(len(non_split)),
        "unique_split_groups"     : int(split_groups.nunique()),
        "avg_chunks_per_group"    : round(float(group_sizes.mean()), 2) if len(group_sizes) else 0,
        "max_chunks_in_group"     : int(group_sizes.max()) if len(group_sizes) else 0,
        "pct_split"               : round(100 * len(split_df) / len(df), 1) if len(df) else 0,
        "chunk_index_denom_dist"  : {str(k): int(v) for k, v in denom_dist.items()},
    }


split_stats = compute_split_stats(df)

print("\n C · SPLIT ANALYSIS")
for k, v in split_stats.items():
    if k != "chunk_index_denom_dist":
        print(f"  {k:<30}: {v}")
print(f"  {'chunk_index_denom_dist':<30}: {split_stats['chunk_index_denom_dist']}")


 C · SPLIT ANALYSIS
  split_chunks                  : 23
  non_split_chunks              : 55
  unique_split_groups           : 9
  avg_chunks_per_group          : 2.56
  max_chunks_in_group           : 5
  pct_split                     : 29.5
  chunk_index_denom_dist        : {'1': 38, '2': 12, '3': 17, '5': 5, '6': 6}


## 7. D · Chunk Type Analysis

In [7]:
def compute_grouped_stats(df: pd.DataFrame) -> dict:
    """Compute counts and average token counts grouped by categorical fields."""

    def group_summary(col: str) -> dict:
        counts = df[col].fillna("(none)").value_counts().to_dict()
        avg    = df.groupby(col)["token_count"].mean().round(2).to_dict()
        return {
            str(k): {"count": int(counts.get(k, 0)), "avg_tokens": round(float(avg.get(k, 0)), 2)}
            for k in counts
        }

    return {
        "by_type"             : group_summary("type"),
        "by_candidate_source" : group_summary("candidate_source"),
        "by_chunking_reason"  : group_summary("chunking_reason"),
        "by_oversize_reason"  : group_summary("oversize_reason"),
    }


grouped = compute_grouped_stats(df)

for section_name, section_data in grouped.items():
    print(f"\n D · {section_name.upper()}")
    print(f"  {'Value':<35} {'Count':>8} {'Avg Tokens':>12}")
    print("  " + "-" * 58)
    for val, metrics in section_data.items():
        print(f"  {str(val):<35} {metrics['count']:>8,} {metrics['avg_tokens']:>12.1f}")


 D · BY_TYPE
  Value                                  Count   Avg Tokens
  ----------------------------------------------------------
  text                                      57        311.0
  table                                     21        159.7

 D · BY_CANDIDATE_SOURCE
  Value                                  Count   Avg Tokens
  ----------------------------------------------------------
  leaf                                      44        339.8
  table                                     21        159.7
  merged_small_chunks                        8        238.0
  non_leaf_large                             5        174.4

 D · BY_CHUNKING_REASON
  Value                                  Count   Avg Tokens
  ----------------------------------------------------------
  kept_in_range                             26        202.8
  split_over_550                            23        458.7
  table_atomic                              21        159.7
  merged_below_180              

## 8. E · Hierarchy Analysis

In [8]:
def compute_hierarchy_stats(df: pd.DataFrame) -> dict:
    """Analyse chunk distribution across document hierarchy levels."""
    by_level = df.groupby("level")["token_count"].agg(["count", "mean"]).round(2)
    by_level.columns = ["count", "avg_tokens"]

    leaf_avg    = df[df["is_leaf_source"] == True]["token_count"].mean()
    nonleaf_avg = df[df["is_leaf_source"] == False]["token_count"].mean()

    parent_title_dist = df["parent_titles_len"].value_counts().sort_index().to_dict()

    return {
        "by_level"                : {
            int(lvl): {"count": int(row["count"]), "avg_tokens": round(float(row["avg_tokens"]), 2)}
            for lvl, row in by_level.iterrows()
        },
        "chunks_with_children"    : int((df["child_count"] > 0).sum()),
        "chunks_without_children" : int((df["child_count"] == 0).sum()),
        "leaf_source_avg_tokens"  : round(float(leaf_avg), 2) if not pd.isna(leaf_avg) else 0,
        "nonleaf_source_avg_tokens": round(float(nonleaf_avg), 2) if not pd.isna(nonleaf_avg) else 0,
        "parent_titles_len_dist"  : {int(k): int(v) for k, v in parent_title_dist.items()},
    }


hierarchy = compute_hierarchy_stats(df)

print("\n E · HIERARCHY")
print(f"  {'Level':<10} {'Count':>8} {'Avg Tokens':>12}")
print("  " + "-" * 32)
for lvl, vals in hierarchy["by_level"].items():
    print(f"  {lvl:<10} {vals['count']:>8,} {vals['avg_tokens']:>12.1f}")
print(f"\n  Chunks WITH children   : {hierarchy['chunks_with_children']:,}")
print(f"  Chunks WITHOUT children: {hierarchy['chunks_without_children']:,}")
print(f"  Leaf source avg tokens : {hierarchy['leaf_source_avg_tokens']}")
print(f"  Non-leaf source avg    : {hierarchy['nonleaf_source_avg_tokens']}")


 E · HIERARCHY
  Level         Count   Avg Tokens
  --------------------------------
  0                 2        127.0
  1                10        382.9
  2                30        296.5
  3                36        225.1

  Chunks WITH children   : 5
  Chunks WITHOUT children: 73
  Leaf source avg tokens : 276.82
  Non-leaf source avg    : 174.4


## 9. F · Block Provenance Analysis

In [9]:
def compute_block_stats(df: pd.DataFrame) -> dict:
    """Analyse the relationship between block count and token count."""
    bc = df["block_count"]

    # Simple correlation proxy: Pearson r via pandas
    corr = df[["block_count", "token_count"]].corr().iloc[0, 1]

    missing_source_ids = int((df["source_block_ids_len"] == 0).sum())

    return {
        "block_count_mean"           : round(float(bc.mean()), 2),
        "block_count_median"         : round(float(bc.median()), 2),
        "block_count_min"            : int(bc.min()),
        "block_count_max"            : int(bc.max()),
        "block_token_pearson_r"      : round(float(corr), 4),
        "chunks_missing_source_ids"  : missing_source_ids,
        "avg_source_block_ids"       : round(float(df["source_block_ids_len"].mean()), 2),
    }


block_stats = compute_block_stats(df)

print("\n F · BLOCK PROVENANCE")
for k, v in block_stats.items():
    print(f"  {k:<35}: {v}")


 F · BLOCK PROVENANCE
  block_count_mean                   : 11.12
  block_count_median                 : 5.5
  block_count_min                    : 0
  block_count_max                    : 85
  block_token_pearson_r              : 0.699
  chunks_missing_source_ids          : 21
  avg_source_block_ids               : 11.12


## 10. G · Overlap Analysis

In [10]:
def compute_overlap_stats(df: pd.DataFrame) -> dict:
    """Analyse overlap token usage across the dataset."""
    prev = df["overlap_prev_tokens"]
    nxt  = df["overlap_next_tokens"]
    return {
        "chunks_with_overlap_prev"     : int((prev > 0).sum()),
        "chunks_with_overlap_next"     : int((nxt  > 0).sum()),
        "avg_overlap_prev_tokens"      : round(float(prev.mean()), 2),
        "avg_overlap_next_tokens"      : round(float(nxt.mean()),  2),
        "total_overlap_prev_tokens"    : int(prev.sum()),
        "total_overlap_next_tokens"    : int(nxt.sum()),
        "total_overlap_tokens"         : int(prev.sum() + nxt.sum()),
    }


overlap_stats = compute_overlap_stats(df)

print("\n G · OVERLAP")
for k, v in overlap_stats.items():
    print(f"  {k:<35}: {v}")


 G · OVERLAP
  chunks_with_overlap_prev           : 14
  chunks_with_overlap_next           : 0
  avg_overlap_prev_tokens            : 7.18
  avg_overlap_next_tokens            : 0.0
  total_overlap_prev_tokens          : 560
  total_overlap_next_tokens          : 0
  total_overlap_tokens               : 560


## 11. H · Page & Section Analysis

In [11]:
def compute_page_section_stats(df: pd.DataFrame) -> dict:
    """Compute chunks-per-page and chunks-per-section distributions."""
    # Page analysis
    page_counts = df["page_num"].value_counts().sort_index()
    page_avg    = df.groupby("page_num")["token_count"].mean().round(2)
    top10_pages = page_counts.head(10).to_dict()

    # Section analysis
    section_counts  = df["section_id"].value_counts()
    top10_sections  = section_counts.head(10).to_dict()

    return {
        "top10_pages_by_chunks"    : {str(k): int(v) for k, v in top10_pages.items()},
        "top10_sections_by_chunks" : {str(k): int(v) for k, v in top10_sections.items()},
        "avg_chunks_per_page"      : round(float(page_counts.mean()), 2),
        "avg_chunks_per_section"   : round(float(section_counts.mean()), 2),
    }


page_sec_stats = compute_page_section_stats(df)

print("\n H · PAGE & SECTION")
print(f"  avg_chunks_per_page    : {page_sec_stats['avg_chunks_per_page']}")
print(f"  avg_chunks_per_section : {page_sec_stats['avg_chunks_per_section']}")
print("\n  Top 10 pages:")
for pg, cnt in page_sec_stats["top10_pages_by_chunks"].items():
    print(f"    page {pg:<8} → {cnt} chunk(s)")
print("\n  Top 10 sections:")
for sid, cnt in page_sec_stats["top10_sections_by_chunks"].items():
    print(f"    {sid[:50]:<50} → {cnt} chunk(s)")


 H · PAGE & SECTION
  avg_chunks_per_page    : 2.29
  avg_chunks_per_section : 1.86

  Top 10 pages:
    page 10       → 2 chunk(s)
    page 11       → 1 chunk(s)
    page 12       → 2 chunk(s)
    page 15       → 5 chunk(s)
    page 23       → 3 chunk(s)
    page 30       → 2 chunk(s)
    page 32       → 1 chunk(s)
    page 33       → 1 chunk(s)
    page 34       → 1 chunk(s)
    page 36       → 3 chunk(s)

  Top 10 sections:
    EMV_Book_1_v4.4_67_PartIV.B1                       → 7 chunk(s)
    EMV_Book_1_v4.4_15_PartI.3                         → 5 chunk(s)
    EMV_Book_1_v4.4_43_PartIII.11.3.2                  → 4 chunk(s)
    EMV_Book_1_v4.4_51_PartIII.12.2.3                  → 4 chunk(s)
    EMV_Book_1_v4.4_44_PartIII.11.3.4                  → 4 chunk(s)
    EMV_Book_1_v4.4_55_PartIII.12.3.2                  → 3 chunk(s)
    EMV_Book_1_v4.4_61_PartIII.12.4                    → 3 chunk(s)
    EMV_Book_1_v4.4_42_PartIII.11.2.2                  → 3 chunk(s)
    EMV_Book_1_v4.4_23_P

## 12. I · Text-Length Diagnostics

In [12]:
def compute_text_diagnostics(df: pd.DataFrame) -> dict:
    """Inspect raw character-level text properties."""
    cc = df["char_count"]

    empty_mask = df["text"].str.strip() == ""

    shortest10 = (
        df.nsmallest(10, "token_count")[["chunk_id", "token_count", "char_count", "title"]]
        .to_dict(orient="records")
    )
    longest10 = (
        df.nlargest(10, "token_count")[["chunk_id", "token_count", "char_count", "title"]]
        .to_dict(orient="records")
    )

    return {
        "avg_char_count"          : round(float(cc.mean()), 2),
        "median_char_count"       : round(float(cc.median()), 2),
        "avg_char_token_ratio"    : round(float(df["char_token_ratio"].mean()), 2),
        "empty_text_chunks"       : int(empty_mask.sum()),
        "top10_shortest"          : shortest10,
        "top10_longest"           : longest10,
    }


text_diag = compute_text_diagnostics(df)

print("\n I · TEXT DIAGNOSTICS")
print(f"  avg_char_count        : {text_diag['avg_char_count']}")
print(f"  median_char_count     : {text_diag['median_char_count']}")
print(f"  avg_char_token_ratio  : {text_diag['avg_char_token_ratio']}")
print(f"  empty_text_chunks     : {text_diag['empty_text_chunks']}")
print("\n  Top 3 shortest:")
for c in text_diag["top10_shortest"][:3]:
    print(f"    {c['chunk_id']} | {c['token_count']} tokens | {c['title'][:40]}")
print("\n  Top 3 longest:")
for c in text_diag["top10_longest"][:3]:
    print(f"    {c['chunk_id']} | {c['token_count']} tokens | {c['title'][:40]}")


 I · TEXT DIAGNOSTICS
  avg_char_count        : 1220.32
  median_char_count     : 958.0
  avg_char_token_ratio  : 4.38
  empty_text_chunks     : 0

  Top 3 shortest:
    EMV_Book_1_v4.4_73_PartIV.B2_text_1 | 13 tokens | Data Elements by Tag
    EMV_Book_1_v4.4_66_PartIV.Annex_A_text_1 | 23 tokens | Removed in Version 4.4
    EMV_Book_1_v4.4_34_Part_II_text_1 | 24 tokens | Removed in Version 4.4

  Top 3 longest:
    EMV_Book_1_v4.4_23_PartI.4.1_text_2 | 586 tokens | Abbreviations
    EMV_Book_1_v4.4_55_PartIII.12.3.2_text_2 | 586 tokens | Using the PSE
    EMV_Book_1_v4.4_15_PartI.3_text_3 | 585 tokens | Definitions


## 13. J · Quality Flags

In [13]:
IMPORTANT_KEYS = ["chunk_id", "section_id", "doc_id", "text", "token_count", "type"]


def compute_quality_flags(df: pd.DataFrame) -> tuple:
    """
    Apply rule-based quality flags to each chunk.
    Returns:
        df_flagged  : DataFrame of chunks that triggered at least one flag
        flag_counts : dict of flag → count
    """
    flag_df = df.copy()

    flag_df["flag_too_small"]      = flag_df["token_count"] < 80
    flag_df["flag_small"]          = (flag_df["token_count"] >= 80) & (flag_df["token_count"] < 180)
    flag_df["flag_ideal"]          = (flag_df["token_count"] >= 350) & (flag_df["token_count"] <= 550)
    flag_df["flag_oversize"]       = flag_df["token_count"] > 550
    flag_df["flag_missing_text"]   = flag_df["text"].str.strip() == ""
    flag_df["flag_missing_meta"]   = flag_df[IMPORTANT_KEYS].isnull().any(axis=1) | \
                                     (flag_df[IMPORTANT_KEYS].astype(str) == "").any(axis=1)

    flag_cols = [c for c in flag_df.columns if c.startswith("flag_")]

    flag_counts = {col.replace("flag_", ""): int(flag_df[col].sum()) for col in flag_cols}

    # Keep rows with at least one True flag, plus the reason columns
    has_any_flag   = flag_df[flag_cols].any(axis=1)
    flagged_subset = flag_df[has_any_flag][["chunk_id", "section_id", "token_count", "type", "title"] + flag_cols].copy()

    # Summarise active flags as a readable string
    def active_flags(row):
        return ", ".join(col.replace("flag_", "") for col in flag_cols if row[col])

    flagged_subset["reasons"] = flagged_subset.apply(active_flags, axis=1)

    return flagged_subset, flag_counts


flagged_df, flag_counts = compute_quality_flags(df)

print("\n J · QUALITY FLAGS")
for flag, cnt in flag_counts.items():
    pct = round(100 * cnt / len(df), 1)
    print(f"  {flag:<25}: {cnt:>6,}  ({pct}%)")
print(f"\n  Total flagged chunks  : {len(flagged_df):,}")


 J · QUALITY FLAGS
  too_small                :     12  (15.4%)
  small                    :     21  (26.9%)
  ideal                    :     20  (25.6%)
  oversize                 :      5  (6.4%)
  missing_text             :      0  (0.0%)
  missing_meta             :      0  (0.0%)

  Total flagged chunks  : 58


## 14. K · Generate Plots

In [14]:
def _save(fig, name: str, output_dir: Path):
    path = output_dir / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  💾 {path.name}")


def generate_plots(df: pd.DataFrame, output_dir: Path):
    """Generate and save all analysis plots."""
    print("\n K · GENERATING PLOTS")

    # 1. Histogram — token_count
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(df["token_count"], bins=50, color="#4A90D9", edgecolor="white", alpha=0.88)
    for thresh, label, color in [
        (80,  "too_small",  "#E74C3C"),
        (180, "small",      "#F39C12"),
        (350, "ideal_low",  "#27AE60"),
        (550, "ideal_high", "#E74C3C"),
    ]:
        ax.axvline(thresh, color=color, linestyle="--", linewidth=1.3, label=str(thresh))
    ax.set_title("Token Count Distribution", fontweight="bold")
    ax.set_xlabel("Token Count")
    ax.set_ylabel("# Chunks")
    ax.legend(title="Thresholds", fontsize=8)
    _save(fig, "01_token_count_histogram", output_dir)

    # 2. Histogram — block_count
    fig, ax = plt.subplots(figsize=(9, 4))
    max_bc = int(df["block_count"].max())
    ax.hist(df["block_count"], bins=min(max_bc + 1, 40), color="#8E44AD", edgecolor="white", alpha=0.85)
    ax.set_title("Block Count Distribution", fontweight="bold")
    ax.set_xlabel("Block Count")
    ax.set_ylabel("# Chunks")
    _save(fig, "02_block_count_histogram", output_dir)

    # 3. Bar — size_band counts
    fig, ax = plt.subplots(figsize=(8, 4))
    band_labels = ["below_100", "between_100_179", "between_180_349", "between_350_550", "above_550"]
    band_values = [
        int((df["token_count"] < 100).sum()),
        int(((df["token_count"] >= 100) & (df["token_count"] < 179)).sum()),
        int(((df["token_count"] >= 180) & (df["token_count"] < 350)).sum()),
        int(((df["token_count"] >= 350) & (df["token_count"] <= 550)).sum()),
        int((df["token_count"] > 550).sum()),
    ]
    colors = ["#E74C3C", "#F39C12", "#27AE60", "#C0392B"]
    bars = ax.bar(band_labels, band_values, color=colors, edgecolor="white")
    ax.bar_label(bars, padding=3)
    ax.set_title("Chunks per Size Band", fontweight="bold")
    ax.set_ylabel("# Chunks")
    plt.xticks(rotation=15, ha="right")
    _save(fig, "03_size_band_counts", output_dir)

    # 4. Bar — chunk type
    fig, ax = plt.subplots(figsize=(7, 4))
    type_counts = df["type"].value_counts()
    bars = ax.bar(type_counts.index, type_counts.values, color=["#3498DB", "#E67E22"], edgecolor="white")
    ax.bar_label(bars, padding=3)
    ax.set_title("Chunks by Type", fontweight="bold")
    ax.set_ylabel("# Chunks")
    _save(fig, "04_chunk_type_counts", output_dir)

    # 5. Bar — level counts
    fig, ax = plt.subplots(figsize=(8, 4))
    level_counts = df["level"].value_counts().sort_index()
    bars = ax.bar(level_counts.index.astype(str), level_counts.values, color="#1ABC9C", edgecolor="white")
    ax.bar_label(bars, padding=3)
    ax.set_title("Chunks per Hierarchy Level", fontweight="bold")
    ax.set_xlabel("Level")
    ax.set_ylabel("# Chunks")
    _save(fig, "05_level_counts", output_dir)

    # 6. Bar — chunking_reason
    fig, ax = plt.subplots(figsize=(10, 4))
    reason_counts = df["chunking_reason"].fillna("(none)").value_counts()
    bars = ax.bar(reason_counts.index, reason_counts.values, color="#2980B9", edgecolor="white")
    ax.bar_label(bars, padding=3)
    ax.set_title("Chunks by Chunking Reason", fontweight="bold")
    ax.set_ylabel("# Chunks")
    plt.xticks(rotation=20, ha="right")
    _save(fig, "06_chunking_reason_counts", output_dir)

    # 7. Scatter — block_count vs token_count
    fig, ax = plt.subplots(figsize=(8, 5))
    sample = df.sample(min(len(df), 1000), random_state=42)   # cap at 1 000 for speed
    ax.scatter(sample["block_count"], sample["token_count"],
               alpha=0.35, s=20, color="#8E44AD")
    ax.set_title("Block Count vs Token Count", fontweight="bold")
    ax.set_xlabel("Block Count")
    ax.set_ylabel("Token Count")
    _save(fig, "07_block_vs_token_scatter", output_dir)

    # 8. Bar — top 10 pages by number of chunks
    fig, ax = plt.subplots(figsize=(10, 4))
    top_pages = df["page_num"].value_counts().head(10).sort_values(ascending=False)
    bars = ax.bar(top_pages.index.astype(str), top_pages.values, color="#E67E22", edgecolor="white")
    ax.bar_label(bars, padding=3)
    ax.set_title("Top 10 Pages by Number of Chunks", fontweight="bold")
    ax.set_xlabel("Page Number")
    ax.set_ylabel("# Chunks")
    _save(fig, "08_top10_pages", output_dir)


generate_plots(df, OUTPUT_DIR)


 K · GENERATING PLOTS
  💾 01_token_count_histogram.png
  💾 02_block_count_histogram.png
  💾 03_size_band_counts.png
  💾 04_chunk_type_counts.png
  💾 05_level_counts.png
  💾 06_chunking_reason_counts.png
  💾 07_block_vs_token_scatter.png
  💾 08_top10_pages.png


## 15. L · Save Outputs

In [15]:
def save_outputs(
    basic       : dict,
    size_bands  : dict,
    split_stats : dict,
    grouped     : dict,
    hierarchy   : dict,
    block_stats : dict,
    overlap     : dict,
    page_sec    : dict,
    text_diag   : dict,
    flag_counts : dict,
    flagged_df  : pd.DataFrame,
    df          : pd.DataFrame,
    output_dir  : Path,
):
    """Persist all statistics and DataFrames to disk."""

    # ── summary_stats.json ────────────────────────────────────────────────
    summary = {
        "A_basic"        : basic,
        "B_size_bands"   : size_bands,
        "C_split"        : split_stats,
        "E_hierarchy"    : hierarchy,
        "F_blocks"       : block_stats,
        "G_overlap"      : overlap,
        "H_page_section" : page_sec,
        "I_text"         : {k: v for k, v in text_diag.items() if k not in ("top10_shortest", "top10_longest")},
        "J_quality_flags": flag_counts,
    }
    summary_path = output_dir / "summary_stats.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"  💾 summary_stats.json")

    # ── grouped_stats.json ────────────────────────────────────────────────
    grouped_all = {
        "D_grouped"   : grouped,
        "top10_shortest": text_diag["top10_shortest"],
        "top10_longest" : text_diag["top10_longest"],
    }
    grouped_path = output_dir / "grouped_stats.json"
    with open(grouped_path, "w", encoding="utf-8") as f:
        json.dump(grouped_all, f, ensure_ascii=False, indent=2)
    print(f"  💾 grouped_stats.json")

    # ── flagged_chunks.csv ────────────────────────────────────────────────
    flagged_path = output_dir / "flagged_chunks.csv"
    flagged_df.to_csv(flagged_path, index=False, encoding="utf-8")
    print(f"  💾 flagged_chunks.csv  ({len(flagged_df):,} rows)")

    # ── chunk_dataframe.csv ───────────────────────────────────────────────
    csv_cols = [
        "chunk_id", "section_id", "chunk_index", "doc_id", "title",
        "section_number", "page_num", "level", "token_count", "char_count",
        "type", "block_count", "is_split", "candidate_source",
        "chunking_reason", "size_band", "overlap_prev_tokens", "overlap_next_tokens",
    ]
    # Only keep columns that exist
    csv_cols = [c for c in csv_cols if c in df.columns]
    df_path  = output_dir / "chunk_dataframe.csv"
    df[csv_cols].to_csv(df_path, index=False, encoding="utf-8")
    print(f"  💾 chunk_dataframe.csv ({len(df):,} rows)")


print("\n L · SAVING OUTPUTS")
save_outputs(
    basic, size_bands, split_stats, grouped,
    hierarchy, block_stats, overlap_stats, page_sec_stats,
    text_diag, flag_counts, flagged_df, df, OUTPUT_DIR
)


 L · SAVING OUTPUTS
  💾 summary_stats.json
  💾 grouped_stats.json
  💾 flagged_chunks.csv  (58 rows)
  💾 chunk_dataframe.csv (78 rows)


## 16. Console Summary

In [16]:
print()
print("╔" + "═" * 56 + "╗")
print("║  EMV CHUNK ANALYSIS — FINAL SUMMARY" + " " * 20 + "║")
print("╠" + "═" * 56 + "╣")
rows = [
    ("Total chunks",         f"{basic['total_chunks']:,}"),
    ("Text / Table",         f"{basic['text_chunks']:,} / {basic['table_chunks']:,}"),
    ("Unique sections",      f"{basic['unique_sections']:,}"),
    ("Avg tokens",           f"{basic['token_count_mean']}"),
    ("Median tokens",        f"{basic['token_count_median']}"),
    ("Split chunks",         f"{split_stats['split_chunks']:,}  ({split_stats['pct_split']}%)"),
    ("Table chunks",         f"{basic['table_chunks']:,}"),
    ("Quality-flagged",      f"{len(flagged_df):,}"),
    ("Empty text chunks",    f"{text_diag['empty_text_chunks']:,}"),
    ("Block↔Token corr (r)", f"{block_stats['block_token_pearson_r']}"),
]
for label, value in rows:
    print(f"║  {label:<30} {value:<23}║")
print("╚" + "═" * 56 + "╝")
print(f"\n✅ All outputs saved to: {OUTPUT_DIR.resolve()}")


╔════════════════════════════════════════════════════════╗
║  EMV CHUNK ANALYSIS — FINAL SUMMARY                    ║
╠════════════════════════════════════════════════════════╣
║  Total chunks                   78                     ║
║  Text / Table                   57 / 21                ║
║  Unique sections                42                     ║
║  Avg tokens                     270.26                 ║
║  Median tokens                  228.0                  ║
║  Split chunks                   23  (29.5%)            ║
║  Table chunks                   21                     ║
║  Quality-flagged                58                     ║
║  Empty text chunks              0                      ║
║  Block↔Token corr (r)           0.699                  ║
╚════════════════════════════════════════════════════════╝

✅ All outputs saved to: /app/src/data/processed/book1/done/analysis
